In [1]:
import os
import json
import pandas as pd

json_dir = "/nfs/hongshu/traces/analysis_json"
rows = []

for fname in os.listdir(json_dir):
    if fname.endswith(".json"):
        cluster = fname.split(".")[0]  # Extract 'clusterX' from filename
        download_path = None
        slab_size = 4
        with open(os.path.join(json_dir, fname), "r") as f:
            try:
                data = json.load(f)
                if cluster.startswith("cluster"):
                    cluster = f"twitter_{cluster}"
                    download_path = f"twitter/{fname[:-14]}"
                    slab_size = 1
                elif cluster.startswith("w"):
                    cluster = f"cp_{cluster}"
                    download_path = f"cloudphysics/{fname[:-14]}"
                else:
                    cluster = f"meta_{'_'.join(cluster.split('_')[:2])}"
                    download_path = f"metaKV/{fname[:-14]}"
                data["trace_name"] = cluster
                data["download_path"] = download_path
                data["file_name"] = fname.split('.')[0] + '.bin'
                data["slab_size"] = slab_size
                rows.append(data)

            except json.JSONDecodeError as e:
                print(f"Error decoding JSON from {fname}: {e}")

df = pd.DataFrame(rows)

In [2]:
df[df['trace_name'] == 'twitter_cluster53']

,number_of_requests,min_req_size,max_req_size,qps,number_of_objects,number_of_req_GiB,number_of_obj_GiB,compulsory_miss_ratio_req,compulsory_miss_ratio_byte,time_span,frequency_mean,trace_name,download_path,file_name,slab_size
146,246508262,10,40575,1331.9155,7263330,1580.5428,10.0239,0.0295,0.0063,185078,33.9387,twitter_cluster53,twitter/cluster53.oracleGeneral.zst,cluster53.bin,1


In [3]:
df['expected_time'] = df['number_of_requests'] / (383887 * 3600)

In [11]:
df.loc[df['expected_time'].idxmax()]

number_of_requests                                    13426570607
min_req_size                                                    9
max_req_size                                                 6988
qps                                                    24228.6458
number_of_objects                                       251815444
number_of_req_GiB                                       2067.6507
number_of_obj_GiB                                          22.932
compulsory_miss_ratio_req                                  0.0188
compulsory_miss_ratio_byte                                 0.0111
time_span                                                  554161
frequency_mean                                            53.3191
trace_name                                      twitter_cluster52
download_path                 twitter/cluster52.oracleGeneral.zst
file_name                                           cluster52.bin
slab_size                                                       1
expected_t

In [4]:
"""
define a function, pass in a wsr, calculate a new field for df, slab_cnt_${wsr}
logic: 
raw_size = wss * wsr * 1024
slab_cnt = int(math.ceil(raw_size / slab_size))
num_slab_for_headers = (7 * slab_cnt + slab_size * 1024 - 1) // (slab_size * 1024)
total_slabs = slab_cnt + num_slab_for_headers
"""

'\ndefine a function, pass in a wsr, calculate a new field for df, slab_cnt_${wsr}\nlogic: \nraw_size = wss * wsr * 1024\nslab_cnt = int(math.ceil(raw_size / slab_size))\nnum_slab_for_headers = (7 * slab_cnt + slab_size * 1024 - 1) // (slab_size * 1024)\ntotal_slabs = slab_cnt + num_slab_for_headers\n'

In [5]:
pd.set_option('display.float_format', '{:.4f}'.format)
df['wss'] = df['number_of_obj_GiB'] + (df['number_of_objects'] * (20 + 32) / (1024 * 1024 * 1024))  # Convert to GiB

In [6]:
import math

def add_slab_cnt_field(df, wsr):
    """
    Adds a new column to df: slab_cnt_{wsr}
    """
    col_name = f"slab_cnt_{wsr}"
    def calc_total_slabs(row):
        raw_size = row['wss'] * wsr * 1024
        slab_size = row['slab_size']
        slab_cnt = int(math.ceil(raw_size / slab_size))
        num_slab_for_headers = (7 * slab_cnt + slab_size * 1024 - 1) // (slab_size * 1024)
        total_slabs = slab_cnt + num_slab_for_headers
        return total_slabs
    df[col_name] = df.apply(calc_total_slabs, axis=1)
    return df

In [7]:
# Assuming df is already defined
df = add_slab_cnt_field(df, 0.005)
df = add_slab_cnt_field(df, 0.05)
df = add_slab_cnt_field(df, 0.01)
df = add_slab_cnt_field(df, 0.1)

In [12]:
print(df[df['trace_name'].str.startswith('twitter')].shape)
print(df[(df['trace_name'].str.startswith('twitter')) & (df['slab_cnt_0.005'] >= 43)].shape[0])
print(df[(df['trace_name'].str.startswith('twitter')) & (df['slab_cnt_0.1'] >= 43)].shape[0])

(50, 21)
24
48


In [31]:
df[df['trace_name'].str.startswith('twitter')].sort_values(by='slab_cnt_0.01', ascending=True)[['trace_name', 'slab_cnt_0.01', 'number_of_requests', 'expected_time']]

,trace_name,slab_cnt_0.01,number_of_requests,expected_time
2,twitter_cluster1,2,6461081324,4.6752
104,twitter_cluster2,5,7226679214,5.2292
141,twitter_cluster47,8,6225239439,4.5045
62,twitter_cluster51,8,6322932555,4.5752
16,twitter_cluster3,9,820307312,0.5936
56,twitter_cluster25,9,10544409258,7.6299
117,twitter_cluster18,10,13062209198,9.4517
43,twitter_cluster7,10,1044513075,0.7558
23,twitter_cluster8,12,1302137879,0.9422
71,twitter_cluster9,14,10646675648,7.7039


In [12]:
import math
raw_size = 0.0176 * 1024 * 0.05
slab_size = 1
slab_cnt = int(math.ceil(raw_size / slab_size))
print(slab_cnt)
num_slab_for_headers = (7 * slab_cnt + slab_size * 1024 - 1) // (slab_size * 1024)
total_slabs = slab_cnt + num_slab_for_headers
rounded_size = total_slabs * slab_size
print(rounded_size)

1
2


In [7]:
trace_names = df['trace_name'].unique()
for trace_name in trace_names:
    if trace_name.startswith("cp"):
        print(f"Trace: {trace_name}")

Trace: cp_w54
Trace: cp_w50
Trace: cp_w98
Trace: cp_w92
Trace: cp_w82
Trace: cp_w12
Trace: cp_w94
Trace: cp_w79
Trace: cp_w100
Trace: cp_w24
Trace: cp_w47
Trace: cp_w39
Trace: cp_w14
Trace: cp_w10
Trace: cp_w18
Trace: cp_w51
Trace: cp_w37
Trace: cp_w09
Trace: cp_w34
Trace: cp_w64
Trace: cp_w97
Trace: cp_w102
Trace: cp_w30
Trace: cp_w22
Trace: cp_w106
Trace: cp_w45
Trace: cp_w46
Trace: cp_w57
Trace: cp_w06
Trace: cp_w91
Trace: cp_w84
Trace: cp_w44
Trace: cp_w59
Trace: cp_w52
Trace: cp_w23
Trace: cp_w61
Trace: cp_w28
Trace: cp_w80
Trace: cp_w20
Trace: cp_w73
Trace: cp_w08
Trace: cp_w26
Trace: cp_w60
Trace: cp_w65
Trace: cp_w68
Trace: cp_w21
Trace: cp_w32
Trace: cp_w88
Trace: cp_w58
Trace: cp_w36
Trace: cp_w70
Trace: cp_w07
Trace: cp_w03
Trace: cp_w49
Trace: cp_w38
Trace: cp_w15
Trace: cp_w105
Trace: cp_w16
Trace: cp_w56
Trace: cp_w01
Trace: cp_w101
Trace: cp_w42
Trace: cp_w81
Trace: cp_w04
Trace: cp_w74
Trace: cp_w25
Trace: cp_w93
Trace: cp_w40
Trace: cp_w72
Trace: cp_w96
Trace: cp_w29
T

In [7]:
df[df['trace_name'] == 'twitter_cluster4']

,number_of_requests,min_req_size,max_req_size,qps,number_of_objects,number_of_req_GiB,number_of_obj_GiB,compulsory_miss_ratio_req,compulsory_miss_ratio_byte,time_span,frequency_mean,trace_name,download_path,file_name,slab_size,wss
150,3448082328,56,179665,5537.6470,106578360,770.5402,13.6594,0.0309,0.0177,622662,32.3526,twitter_cluster4,twitter/cluster4.oracleGeneral.zst,cluster4.oracleGeneral.zst,1,18.8209


The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


In [29]:
[v for v in df['trace_name'].values.tolist() if v.startswith("meta")]

['meta_202312_kv',
 'meta_meta_kvcache',
 'meta_202210_kv',
 'meta_202206_kv',
 'meta_202401_kv']

In [28]:
df.to_csv("trace_info.csv", index=False)

In [17]:
"""
Slab classes start at 64 bytes and exponentially increase in size by a factor of 1.07 up to 1 MB, aligned on
4-byte boundaries3
"""
import numpy as np

df['min_item_size'] = np.maximum(df['min_req_size'], 24) + 20 + 32
df['max_item_size'] = df['max_req_size'] + 20 + 32


In [18]:
print(f"min item size, min{df['min_item_size'].min()}, max {df['min_item_size'].max()}")
print(f"max item size, min{df['max_item_size'].min()}, max {df['max_item_size'].max()}")

min item size, min76, max 120
max item size, min94, max 1010406


In [37]:
df[df['trace_name'].str.startswith('twitter')].sort_values(by=["compulsory_miss_ratio_req"], ascending=False)[['trace_name', 'compulsory_miss_ratio_req']]

,trace_name,compulsory_miss_ratio_req
121,twitter_cluster31,0.9383
105,twitter_cluster39,0.9383
114,twitter_cluster38,0.9383
14,twitter_cluster21,0.9375
24,twitter_cluster13,0.6279
138,twitter_cluster10,0.4999
157,twitter_cluster45,0.2876
148,twitter_cluster23,0.2703
80,twitter_cluster37,0.2200
72,twitter_cluster22,0.1988


In [19]:
import numpy as np


def find_slab_classes(min_size=64, max_size=1024 * 1024, step_factor=1.07):
    sizes = []
    size = min_size

    while size <= max_size:
        # Align to 4 bytes
        aligned_size = int(np.ceil(size / 4.0)) * 4
        if len(sizes) == 0 or aligned_size != sizes[-1]:
            sizes.append(aligned_size)
        size *= step_factor
    
    return sizes

In [20]:
classes = find_slab_classes(72, 1 * 1024 * 1024, 1.25)
print("Slab classes:", classes)
print(len(classes), "classes")

Slab classes: [72, 92, 116, 144, 176, 220, 276, 344, 432, 540, 672, 840, 1048, 1312, 1640, 2048, 2560, 3200, 4000, 5000, 6248, 7808, 9760, 12200, 15248, 19060, 23824, 29780, 37224, 46532, 58164, 72704, 90880, 113596, 141996, 177496, 221868, 277336, 346668, 433336, 541668, 677088, 846356]
43 classes


In [14]:
max_sizes = [val + 32 + 20 for val in df['max_req_size'].values.tolist()]
print("above 512kb: ", sum([1 for val in max_sizes if val > 512 * 1024]))
print("above 256kb: ", sum([1 for val in max_sizes if val > 256 * 1024]))
print("above 128kb: ", sum([1 for val in max_sizes if val > 128 * 1024]))
print("above 64kb: ", sum([1 for val in max_sizes if val > 64 * 1024]))
print("above 32kb: ", sum([1 for val in max_sizes if val > 32 * 1024]))
print("above 16kb: ", sum([1 for val in max_sizes if val > 16 * 1024]))
print("above 8kb: ", sum([1 for val in max_sizes if val > 8 * 1024]))

above 512kb:  3
above 256kb:  5
above 128kb:  11
above 64kb:  13
above 32kb:  18
above 16kb:  23
above 8kb:  29


In [13]:
1024 * 1024 * 0.5

524288.0

In [ ]:
"""
0.766    cluster26.oracleGeneral.zst
0.802    cluster10.oracleGeneral.zst [compulsory misses]
1.020    cluster50.oracleGeneral.zst done
1.565    cluster53.oracleGeneral.zst done
1.660    cluster45.oracleGeneral.zst
4.300    cluster3.oracleGeneral.zst
5.012    cluster49.oracleGeneral.zst
5.261    cluster35.oracleGeneral.zst
5.826    cluster7.oracleGeneral.zst
6.238    cluster13.oracleGeneral.zst [compulsory misses]
7.355    cluster8.oracleGeneral.zst
10.172   cluster31.oracleGeneral.zst [compulsory misses]
11.138   cluster48.oracleGeneral.zst
11.361   cluster39.oracleGeneral.zst [compulsory misses]
11.966   cluster38.oracleGeneral.zst [compulsory misses]
13.156   cluster21.oracleGeneral.zst [compulsory misses]
14.107   cluster30.oracleGeneral.zst
15.205   cluster22.oracleGeneral.zst
16.460   cluster20.oracleGeneral.zst
18.653   cluster24.oracleGeneral.zst
18.742   cluster34.oracleGeneral.zst
18.990   cluster19.oracleGeneral.zst
19.676   cluster37.oracleGeneral.zst
20.288   cluster42.oracleGeneral.zst
21.045   cluster41.oracleGeneral.zst
21.148   cluster4.oracleGeneral.zst
21.176   cluster14.oracleGeneral.zst
22.523   cluster12.oracleGeneral.zst
22.833   cluster11.oracleGeneral.zst
23.653   cluster2.oracleGeneral.zst
25.245   cluster40.oracleGeneral.zst
25.697   cluster1.oracleGeneral.zst, careful
26.560   cluster25.oracleGeneral.zst
30.276   cluster28.oracleGeneral.zst
30.964   cluster32.oracleGeneral.zst
33.114   cluster47.oracleGeneral.zst
33.707   cluster51.oracleGeneral.zst
34.875   cluster23.oracleGeneral.zst
35.066   cluster33.oracleGeneral.zst
36.377   cluster36.oracleGeneral.zst
38.071   cluster44.oracleGeneral.zst
40.701   cluster46.oracleGeneral.zst
44.324   cluster15.oracleGeneral.zst
46.665   cluster29.oracleGeneral.zst
48.676   cluster6.oracleGeneral.zst
48.724   cluster17.oracleGeneral.zst
54.769   cluster9.oracleGeneral.zst
68.743   cluster16.oracleGeneral.zst
70.116   cluster43.oracleGeneral.zst
75.081   cluster5.oracleGeneral.zst  slow
76.434   cluster18.oracleGeneral.zst  slow
92.445   cluster52.oracleGeneral.zst
97.133   cluster54.oracleGeneral.zst
130.552  cluster27.oracleGeneral.zst
"""